<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 2.8: Memories
**Prev: [Enums, Decoders and Bundles](2.7_enums_and_bundles.ipynb)**<br>
**Next: [Generators: Parameters](3.1_parameters.ipynb)**

## Motivation
Registers are fine for a few values, but a processor needs memories: a register file, an instruction
memory, a data memory, caches. A memory is an array of words with *ports*, each of which reads or
writes one word per cycle. This module shows the memories Chisel provides and how they behave in
simulation:

- `Mem`: reads are combinational (the data comes back in the same cycle);
- `SyncReadMem`: reads take one clock cycle, like the SRAM blocks of FPGAs and ASICs;
- write masks, to write only some bytes of a word;
- `SRAM`, a memory described by its ports, and initializing a memory from a file;
- a register file with two read ports, which you will reuse when building a RISC-V processor.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._

---
# Mem: combinational read

`Mem(n, t)` is `n` words of type `t`. `mem(addr)` (or `mem.read(addr)`) reads a word and
`mem.write(addr, data)` writes one. Writes take effect at the next rising clock edge, like a
register; reads of a `Mem` are combinational, so the data is available in the same cycle as the
address.

In [ ]:
// the ports of the memories below: one write port and one read port
class MemPorts extends Bundle {
  val wen   = Input(Bool())
  val waddr = Input(UInt(4.W))
  val wdata = Input(UInt(8.W))
  val raddr = Input(UInt(4.W))
  val rdata = Output(UInt(8.W))
}

class CombMemory extends Module {
  val io  = IO(new MemPorts)
  val mem = Mem(16, UInt(8.W))
  when(io.wen) { mem.write(io.waddr, io.wdata) }
  io.rdata := mem(io.raddr)
}

simulate(new CombMemory) { c =>
  c.io.wen.poke(true.B)
  c.io.waddr.poke(3.U)
  c.io.wdata.poke(42.U)
  c.io.raddr.poke(3.U)
  c.clock.step() // the write happens at this clock edge
  c.io.wen.poke(false.B)
  c.io.rdata.expect(42.U) // and the read sees it without another edge
  c.io.wdata.poke(7.U)
  c.io.wen.poke(true.B)   // overwrite address 3 ...
  c.io.rdata.expect(42.U) // ... but until the edge, the read returns the old value
  c.clock.step()
  c.io.rdata.expect(7.U)
}
println("SUCCESS!!")

> The contents of a memory are unknown until they are written (simulators may start them at zero or
> at random values), so never rely on the initial value.

A combinational read is convenient (a single-cycle processor reads its register file and its memory
in the same cycle as it decodes), but large memories cannot be built that way. SRAM macros and FPGA
block RAMs register the address (or the data), so their reads take one cycle.

---
# SyncReadMem: one-cycle read

`SyncReadMem` reads synchronously: the address is sampled at the clock edge and the data is valid
after it. Compare with `CombMemory`:

In [ ]:
class SyncMemory extends Module {
  val io  = IO(new MemPorts)
  val mem = SyncReadMem(16, UInt(8.W))
  when(io.wen) { mem.write(io.waddr, io.wdata) }
  io.rdata := mem.read(io.raddr)
}

simulate(new SyncMemory) { c =>
  c.io.wen.poke(true.B)
  c.io.waddr.poke(3.U)
  c.io.wdata.poke(42.U)
  c.clock.step()
  c.io.wen.poke(false.B)
  c.io.raddr.poke(3.U)
  c.clock.step()          // the read address is sampled here
  c.io.rdata.expect(42.U) // the data appears one cycle after the address
}
println("SUCCESS!!")

What does a read return when the same address is written in the same cycle? A `SyncReadMem` takes
the answer as an optional third argument:

| `SyncReadMem(n, t, ...)` | A read of the address being written returns |
| :--- | :--- |
| `SyncReadMem.ReadFirst` | the old data |
| `SyncReadMem.WriteFirst` | the new data |
| `SyncReadMem.Undefined` (the default) | either: do not rely on it |

The argument is a request to the tools that build the memory, not a promise. Let's check what this
toolchain does: the test below writes and reads address 5 in the same cycle with both policies.

In [ ]:
class ReadUnderWrite(ruw: SyncReadMem.ReadUnderWrite) extends Module {
  val io = IO(new Bundle {
    val wen   = Input(Bool())
    val addr  = Input(UInt(4.W))
    val wdata = Input(UInt(8.W))
    val rdata = Output(UInt(8.W))
  })
  val mem = SyncReadMem(16, UInt(8.W), ruw)
  when(io.wen) { mem.write(io.addr, io.wdata) }
  io.rdata := mem.read(io.addr)
}

for (ruw <- Seq(SyncReadMem.ReadFirst, SyncReadMem.WriteFirst)) {
  simulate(new ReadUnderWrite(ruw)) { c =>
    c.io.addr.poke(5.U)
    c.io.wen.poke(true.B)
    c.io.wdata.poke(1.U)
    c.clock.step()       // address 5 now holds 1
    c.io.wdata.poke(2.U) // write 2 and read address 5 in the same cycle
    c.clock.step()
    println(s"$ruw: the read returned ${c.io.rdata.peek().litValue} (old data 1, new data 2)")
    assert(c.io.rdata.peek().litValue == 2) // what firtool 1.160.0 builds, whatever the policy
  }
}

Both return the new data. `firtool` (version 1.160.0 here) builds every `SyncReadMem` the same way: it
registers the read address and reads the array after the clock edge, when the write has already
happened. So the policy is lost on the way to Verilog, and an FPGA or ASIC memory may behave
differently again. If a design needs one behavior, build it explicitly: for example, compare the read
and write addresses and forward the write data (or keep the old word in a register).

---
# Write masks

When the word type is a `Vec`, a write can take a mask with one `Bool` per element, and only the
elements whose mask bit is set are written. A memory of `Vec(4, UInt(8.W))` is a 32-bit memory
with a byte mask: exactly what a processor needs for byte (`sb`) and halfword (`sh`) stores.

In [ ]:
class ByteMemory extends Module {
  val io = IO(new Bundle {
    val wen   = Input(Bool())
    val addr  = Input(UInt(4.W))
    val wdata = Input(Vec(4, UInt(8.W)))
    val mask  = Input(Vec(4, Bool()))
    val rdata = Output(Vec(4, UInt(8.W)))
  })
  val mem = Mem(16, Vec(4, UInt(8.W)))
  when(io.wen) { mem.write(io.addr, io.wdata, io.mask) }
  io.rdata := mem(io.addr)
}

simulate(new ByteMemory) { c =>
  c.io.addr.poke(0.U)
  c.io.wen.poke(true.B)
  for (i <- 0 until 4) { c.io.wdata(i).poke((0x10 + i).U); c.io.mask(i).poke(true.B) }
  c.clock.step() // the word is 13 12 11 10
  for (i <- 0 until 4) { c.io.wdata(i).poke(0xff.U); c.io.mask(i).poke((i == 2).B) }
  c.clock.step() // only byte 2 is written
  c.io.wen.poke(false.B)
  for ((expected, i) <- Seq(0x10, 0x11, 0xff, 0x13).zipWithIndex) c.io.rdata(i).expect(expected.U)
}
println("SUCCESS!!")

<span style="color:red">**Exercise: Store Unit**</span><br>
RISC-V stores write a byte (`sb`), a halfword (`sh`) or a word (`sw`) at a byte address. Complete
`StoreUnit`, a 16-word memory that takes a byte address `addr`, a size (`0` for a byte, `1` for a
halfword, `2` for a word) and the value to store in the low bits of `wdata`. Bits 1-0 of the address
select the byte within the word (little-endian: byte 0 holds bits 7-0 of the word), and bits 5-2
select the word. Stores are aligned: a halfword starts at an even address and a word at a multiple of 4.

Compute `mask` (which bytes to write) and `lanes` (the value moved to the bytes where it belongs).
For example, `sb` of `0xab` at address 6 writes byte 2 of word 1: `mask` is `false, false, true,
false` and `lanes(2)` is `0xab`.

In [ ]:
class StoreUnit extends Module {
  val io = IO(new Bundle {
    val wen   = Input(Bool())
    val addr  = Input(UInt(6.W)) // byte address
    val size  = Input(UInt(2.W)) // 0: byte, 1: halfword, 2: word
    val wdata = Input(UInt(32.W))
    val raddr = Input(UInt(4.W)) // word address of the read port
    val rdata = Output(UInt(32.W))
  })
  val mem    = Mem(16, Vec(4, UInt(8.W)))
  val offset = io.addr(1, 0)
  val mask   = Wire(Vec(4, Bool()))
  val lanes  = Wire(Vec(4, UInt(8.W)))

  ???

  when(io.wen) { mem.write(io.addr(5, 2), lanes, mask) }
  io.rdata := mem(io.raddr).asUInt
}

In [ ]:
simulate(new StoreUnit) { c =>
  val model          = Array.fill(64)(0) // the memory, byte by byte
  def wordOf(w: Int) = (0 until 4).map(i => BigInt(model(4 * w + i)) << (8 * i)).sum
  val rnd            = new scala.util.Random(1)
  // the memory starts with random values: clear it with word stores first
  c.io.wen.poke(true.B)
  c.io.size.poke(2.U)
  c.io.wdata.poke(0.U)
  for (word <- 0 until 16) { c.io.addr.poke((4 * word).U); c.clock.step() }
  for (_ <- 0 until 200) {
    val size = rnd.nextInt(3)
    val addr = rnd.nextInt(64) & ~((1 << size) - 1) // aligned
    val data = rnd.nextLong() & 0xffffffffL
    c.io.wen.poke(true.B)
    c.io.addr.poke(addr.U)
    c.io.size.poke(size.U)
    c.io.wdata.poke(data.U)
    c.clock.step()
    for (i <- 0 until (1 << size)) model(addr + i) = ((data >> (8 * i)) & 0xff).toInt
    // check the word just written, so a wrong mask is caught at the store that caused it
    val word = addr >> 2
    c.io.raddr.poke(word.U)
    c.io.rdata.expect(wordOf(word).U, f"store of size $size at 0x$addr%x")
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  // the bytes the store covers, before shifting: 0001, 0011 or 1111
  val sizeMask = MuxLookup(io.size, "b1111".U(4.W))(Seq(0.U -&gt; "b0001".U(4.W), 1.U -&gt; "b0011".U(4.W)))
  mask  := (sizeMask &lt;&lt; offset)(3, 0).asBools
  // move the data up by `offset` bytes
  lanes := (io.wdata &lt;&lt; (offset ## 0.U(3.W)))(31, 0).asTypeOf(Vec(4, UInt(8.W)))
</pre></article></div></section></div>

One caveat about masks. A memory can have several write ports, but `firtool` (version 1.160.0 here,
and 1.163.0 too; reported as [llvm/circt#11591](https://github.com/llvm/circt/issues/11591)) drops the write masks of a memory that has a masked write port and a write port
**without** a mask: each masked write then writes every byte. A constant mask of all ones counts as
no mask (it is optimized away). It happens to a `Mem` and a `SyncReadMem` alike. The Chisel side is
fine (the FIRRTL still has the mask); the masks disappear when `firtool` turns the memory into
Verilog. Write ports whose masks are all signals keep them. The simplest rule: give a memory that
needs a mask a single write port, with `Mux`es that select its address, data and mask from the
sources (all ones for a source that writes whole words), as the RAM of the Module 5 processor does
for its program loader and the processor's stores. Here, a second port without a mask, never even
used, is enough to lose the mask:

In [ ]:
// the ByteMemory above, plus a second write port, without a mask
class TwoWritePorts extends Module {
  val io = IO(new Bundle {
    val wen   = Input(Bool())
    val mask  = Input(Vec(4, Bool()))
    val wdata = Input(Vec(4, UInt(8.W)))
    val other = Input(Bool()) // never enabled below: the unmasked port's existence alone drops the mask
    val rdata = Output(Vec(4, UInt(8.W)))
  })
  val mem = Mem(16, Vec(4, UInt(8.W)))
  when(io.wen) { mem.write(0.U, io.wdata, io.mask) }
  when(io.other) { mem.write(1.U, io.wdata) }
  io.rdata := mem(0.U)
}

simulate(new TwoWritePorts) { c =>
  def pokeBytes(bytes: Seq[Int]) = for ((b, i) <- bytes.zipWithIndex) c.io.wdata(i).poke(b.U)
  c.io.other.poke(false.B)
  c.io.wen.poke(true.B)
  c.io.mask.foreach(_.poke(true.B))
  pokeBytes(Seq(0x44, 0x33, 0x22, 0x11)) // 0x11223344
  c.clock.step()
  c.io.mask(0).poke(false.B) // now write only bytes 2 and 3 of 0xaabbccdd
  c.io.mask(1).poke(false.B)
  pokeBytes(Seq(0xdd, 0xcc, 0xbb, 0xaa))
  c.clock.step()
  val bytes    = c.io.rdata.map(_.peek().litValue.toInt)
  val maskLost = bytes == Seq(0xdd, 0xcc, 0xbb, 0xaa) // 0xaabb3344 if the mask had been kept
  println(bytes.reverse.map(b => f"$b%02x").mkString("0x", "", "") + (if (maskLost) ": the mask was lost" else ""))
  assert(maskLost, "firtool now keeps the mask: this section can be updated")
}

---
# SRAM: a memory described by its ports

`SRAM(size, type, numReadPorts, numWritePorts, numReadwritePorts)` builds a synchronous-read memory
(like a `SyncReadMem`) and returns its ports as a bundle: `readPorts`, `writePorts` and `readwritePorts`, each port with an
`address`, an `enable` and its data. Describing a memory by its ports is how memory compilers and
FPGA tools see memories. The read port's `enable` saves power when the data is not needed.

In [ ]:
class TwoPortSram extends Module {
  val io   = IO(new MemPorts)
  val sram = SRAM(16, UInt(8.W), 1, 1, 0) // 1 read port, 1 write port, 0 read-write ports
  sram.writePorts(0).enable  := io.wen
  sram.writePorts(0).address := io.waddr
  sram.writePorts(0).data    := io.wdata
  sram.readPorts(0).enable   := true.B
  sram.readPorts(0).address  := io.raddr
  io.rdata                   := sram.readPorts(0).data
}

simulate(new TwoPortSram) { c =>
  c.io.wen.poke(true.B)
  c.io.waddr.poke(9.U)
  c.io.wdata.poke(99.U)
  c.clock.step()
  c.io.wen.poke(false.B)
  c.io.raddr.poke(9.U)
  c.clock.step() // synchronous read, as for a SyncReadMem
  c.io.rdata.expect(99.U)
}
println("SUCCESS!!")

`SRAM` also has variants that take a `HexMemoryFile` or `BinaryMemoryFile` to load the memory when
the simulation starts. In Chisel 7.16.0, `SRAM` accepts that file and silently ignores it, in the
default implementation above and in the SRAM black-box flow alike. To initialize a memory, use the
method below.

---
# Initializing a memory from a file

`chisel3.util.experimental.loadMemoryFromFileInline(mem, path)` loads a `Mem` or `SyncReadMem` from a
file; each line of a hex file holds one word, and the optional third argument
`MemoryLoadFileType.Binary` selects binary instead. It only *emits* a Verilog `$readmemh` (or
`$readmemb`) directive with the file's path: the file is read when the simulation starts, not when
Chisel runs. So the file must exist by then, and its path must be valid from the simulator's working
directory, which ChiselSim chooses: use an absolute path.

There is one more catch in simulation. By default ChiselSim fills every memory with random values at
start (to catch designs that read memory before writing it), and that happens *after* `$readmemh`,
overwriting the file's contents. Pass `randomization = Randomization.random.copy(memories = false)`
to `simulate` to turn it off for memories only (registers still start random, which catches a missing
reset).

In [ ]:
import chisel3.util.experimental.loadMemoryFromFileInline
import chisel3.simulator.Randomization

val romFile  = os.pwd / "build" / "rom.hex"
val romWords = Seq(0x11, 0x22, 0x33, 0x44)
os.write.over(romFile, romWords.map(w => f"$w%02x\n").mkString, createFolders = true) // one hex word per line

class Rom(path: String) extends Module {
  val io = IO(new Bundle {
    val addr = Input(UInt(2.W))
    val data = Output(UInt(8.W))
  })
  val mem = Mem(4, UInt(8.W))
  loadMemoryFromFileInline(mem, path)
  io.data := mem(io.addr)
}

println(getVerilog(new Rom(romFile.toString)).split("\n").filter(_.contains("readmem")).mkString("\n"))
simulate(new Rom(romFile.toString), randomization = Randomization.random.copy(memories = false)) { c =>
  for (i <- 0 until 4) {
    c.io.addr.poke(i.U)
    c.io.data.expect(romWords(i).U)
  }
}
println("SUCCESS!!")

---
# Register file

A RISC-V processor has 32 registers `x0`-`x31` of 32 bits. Each instruction reads up to two of them
(`rs1` and `rs2`) and writes at most one (`rd`), so the register file has two combinational read
ports and one write port. Register `x0` is special: it always reads as zero, and writes to it are
ignored.

ChiselSim's `peek` and `poke` only reach the ports of the module under test, not the contents of
a memory inside it. To check the whole register file, the module also has a *debug* read port,
`dbgAddr`/`dbgData`, that tests (and later, debuggers) can use without disturbing the processor's
ports.

<span style="color:red">**Exercise: Register File**</span><br>
Complete `RegFile`: the two read ports and the debug port read the register at their address (0 for
`x0`), and a write with `wen` set stores `wdata` into register `rd` at the clock edge, unless `rd`
is 0. A read in the same cycle as a write to the same register returns the old value; the new value
is visible from the next cycle.

In [ ]:
class RegFile extends Module {
  val io = IO(new Bundle {
    val rs1     = Input(UInt(5.W))
    val rs2     = Input(UInt(5.W))
    val rs1Data = Output(UInt(32.W))
    val rs2Data = Output(UInt(32.W))
    val wen     = Input(Bool())
    val rd      = Input(UInt(5.W))
    val wdata   = Input(UInt(32.W))
    val dbgAddr = Input(UInt(5.W))
    val dbgData = Output(UInt(32.W))
  })
  val regs = Mem(32, UInt(32.W))

  ???
}

The test compares the register file with a Scala model (an array of 32 values) on random reads and
writes. It uses a fixed seed, so a failure is the same every time you run it. Writes to `x0` are
made more frequent than random, since that is the case most likely to be wrong. The registers start
with random values in simulation (as in hardware), so the test first writes every one of them.

In [ ]:
simulate(new RegFile) { c =>
  val model = Array.fill(32)(BigInt(0))
  val rnd   = new scala.util.Random(2)
  // the registers start with random values: give each one a known value first
  c.io.wen.poke(true.B)
  for (r <- 1 until 32) {
    model(r) = BigInt(32, rnd)
    c.io.rd.poke(r.U)
    c.io.wdata.poke(model(r).U)
    c.clock.step()
  }
  def reg() = if (rnd.nextInt(8) == 0) 0 else rnd.nextInt(32)
  for (cycle <- 0 until 500) {
    val (rs1, rs2, rd) = (reg(), reg(), reg())
    val wen            = rnd.nextBoolean()
    val wdata          = BigInt(32, rnd)
    c.io.rs1.poke(rs1.U)
    c.io.rs2.poke(rs2.U)
    c.io.rd.poke(rd.U)
    c.io.wen.poke(wen.B)
    c.io.wdata.poke(wdata.U)
    c.io.rs1Data.expect(model(rs1).U, s"cycle $cycle: x$rs1")
    c.io.rs2Data.expect(model(rs2).U, s"cycle $cycle: x$rs2")
    c.clock.step()
    if (wen && rd != 0) model(rd) = wdata
  }
  c.io.wen.poke(false.B)
  for (r <- 0 until 32) {
    c.io.dbgAddr.poke(r.U)
    c.io.dbgData.expect(model(r).U, s"x$r")
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  def read(addr: UInt): UInt = Mux(addr === 0.U, 0.U, regs(addr))
  io.rs1Data := read(io.rs1)
  io.rs2Data := read(io.rs2)
  io.dbgData := read(io.dbgAddr)
  when(io.wen &amp;&amp; io.rd =/= 0.U) {
    regs.write(io.rd, io.wdata)
  }
</pre></article></div></section></div>

You will use this register file again when you build a RISC-V processor.

---
# You're done!

[Return to the top.](#top)